In [1]:
# ============================================================
# INSTALL
# ============================================================
!pip install transformers datasets accelerate torch -q

In [2]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import torch
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorWithPadding
)

device = torch.device("cuda:0")

MAX_LENGTH = 256
MAX_NEW_TOKEN = 50
BATCH_SIZE = 4
EPOCHS = 1
OUTPUT_SFT_MODEL_PATH = "out/sft_model"
OUTPUT_PPO_MODEL_PATH = "out/ppo_model"
OUTPUT_DPO_MODEL_PATH = "out/dpo_model"

os.makedirs(OUTPUT_SFT_MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PPO_MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DPO_MODEL_PATH, exist_ok=True)

/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
def tokenize_pair(tokenizer, prompt, response):

    p_enc = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)

    full_enc = tokenizer(
        prompt + response,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = full_enc.input_ids.squeeze(0)
    attention_mask = full_enc.attention_mask.squeeze(0)

    prompt_len = len(p_enc["input_ids"])

    return input_ids, attention_mask, prompt_len

def compute_logprob(model, input_ids, attention_mask):

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    shift_log_probs = log_probs[:, :-1, :]
    labels = input_ids[:, 1:]
    shift_attention_mask = attention_mask[:, 1:] # Mask for tokens whose logprobs are computed

    token_logps = shift_log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    # Mask out the log probabilities of padding tokens
    masked_token_logps = token_logps * shift_attention_mask

    return masked_token_logps.sum(dim=1)

def generate_response(model, tokenizer, prompts, attention_mask):

    with torch.no_grad():

        outputs = model.generate(
            input_ids=prompts,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKEN,
            do_sample=True,
            top_p=0.8,         # Slightly more restrictive
            temperature=0.5,   # Lower temperature = "safer" distributions
            pad_token_id=tokenizer.eos_token_id,
            epsilon_cutoff=1e-6 # Some versions of transformers support this
        )

    # Compute attention_mask for generated sequences
    attention_mask = (outputs != tokenizer.pad_token_id).long()

    return outputs, attention_mask

In [4]:
dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_pref = dataset["train"].select(range(2000))
val_pref = dataset["test"].select(range(500))

In [5]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # Set padding side to left for decoder-only models

/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
sft_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda:0")

/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/nam/projects/hoang/Advanced-ASM1/rlhf/lib/python3.13/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
def sft_tokenize(example):

    tokens = tokenizer(
        example["prompt"] + example["chosen"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [8]:
sft_train_dataset = train_pref.map(
    sft_tokenize,
    remove_columns=["prompt","chosen","rejected"]
)

sft_val_dataset = val_pref.map(
    sft_tokenize,
    remove_columns=["prompt","chosen","rejected"]
)

In [9]:
collator = DataCollatorWithPadding(tokenizer)

sft_train_loader = DataLoader(
    sft_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator
)

sft_val_loader = DataLoader(
    sft_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)

In [10]:
optimizer = torch.optim.AdamW(sft_model.parameters(), lr=1e-5)

for epoch in range(EPOCHS):
    sft_model.train()
    train_total = 0

    for step, batch in enumerate(sft_train_loader):

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = sft_model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_total += loss.item()

    train_loss = train_total / len(sft_train_loader)

    sft_model.eval()
    val_total = 0

    with torch.no_grad():
        for step, batch in enumerate(sft_val_loader):

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = sft_model(**batch)
            loss = outputs.loss

            val_total += loss.item()

    val_loss = val_total / len(sft_val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch 1/1 - Train Loss: 2.0028, Val Loss: 1.7520


In [11]:
sft_model.save_pretrained(OUTPUT_SFT_MODEL_PATH)
tokenizer.save_pretrained(OUTPUT_SFT_MODEL_PATH)

('out/sft_model/tokenizer_config.json',
 'out/sft_model/special_tokens_map.json',
 'out/sft_model/vocab.json',
 'out/sft_model/merges.txt',
 'out/sft_model/added_tokens.json',
 'out/sft_model/tokenizer.json')

In [12]:
class RewardModel(nn.Module):

    def __init__(self, base):
        super().__init__()

        self.base = base
        self.head = nn.Linear(base.config.n_embd,1)

    def forward(self,input_ids,attention_mask):

        out = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden = out.hidden_states[-1]

        pooled = (hidden * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)

        return self.head(pooled).squeeze(-1)

In [13]:
def prepare_reward(example):

    c_ids,c_mask,_ = tokenize_pair(tokenizer,example["prompt"],example["chosen"])
    r_ids,r_mask,_ = tokenize_pair(tokenizer,example["prompt"],example["rejected"])

    return {
        "chosen_ids":c_ids,
        "chosen_mask":c_mask,
        "rejected_ids":r_ids,
        "rejected_mask":r_mask
    }

In [14]:
reward_train_dataset = train_pref.map(
    prepare_reward,
    remove_columns=["prompt","chosen","rejected"]
)

reward_val_dataset = val_pref.map(
    prepare_reward,
    remove_columns=["prompt","chosen","rejected"]
)

In [15]:
class RewardCollator:

    def __call__(self, batch):

        return {

            "chosen_ids": torch.stack(
                [torch.tensor(x["chosen_ids"]) for x in batch]
            ),

            "chosen_mask": torch.stack(
                [torch.tensor(x["chosen_mask"]) for x in batch]
            ),

            "rejected_ids": torch.stack(
                [torch.tensor(x["rejected_ids"]) for x in batch]
            ),

            "rejected_mask": torch.stack(
                [torch.tensor(x["rejected_mask"]) for x in batch]
            ),
        }

In [16]:
reward_train_loader = DataLoader(
    reward_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=RewardCollator()
)

reward_val_loader = DataLoader(
    reward_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=RewardCollator()
)

In [17]:
base = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")

reward_model = RewardModel(base).to(device)

optimizer = torch.optim.AdamW(reward_model.parameters(), lr=1e-6)

loss_fn = nn.BCEWithLogitsLoss()

best_val = float("inf")

for epoch in range(EPOCHS):
    reward_model.train()
    train_total = 0

    for step, batch in enumerate(reward_train_loader):

        c_ids = batch["chosen_ids"].to(device)
        c_mask = batch["chosen_mask"].to(device)

        r_ids = batch["rejected_ids"].to(device)
        r_mask = batch["rejected_mask"].to(device)

        c = reward_model(c_ids, c_mask)
        r = reward_model(r_ids, r_mask)

        loss = loss_fn(c - r, torch.ones_like(c))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_total += loss.item()

        if step % 10 == 0:
            print(
                f"[Epoch {epoch+1}/{EPOCHS}] "
                f"[Train Step {step}/{len(reward_train_loader)}] "
                f"Loss: {loss.item():.4f} | "
                f"Chosen mean: {c.mean().item():.4f} | "
                f"Rejected mean: {r.mean().item():.4f}"
            )

    train_loss = train_total / len(reward_train_loader)

    # -----------------------
    # VALIDATION
    # -----------------------
    reward_model.eval()
    val_total = 0

    with torch.no_grad():

        for step, batch in enumerate(reward_val_loader):

            c_ids = batch["chosen_ids"].to(device)
            c_mask = batch["chosen_mask"].to(device)

            r_ids = batch["rejected_ids"].to(device)
            r_mask = batch["rejected_mask"].to(device)

            c = reward_model(c_ids, c_mask)
            r = reward_model(r_ids, r_mask)

            loss = loss_fn(c - r, torch.ones_like(c))

            val_total += loss.item()

            if step % 10 == 0:
                print(
                    f"[Epoch {epoch+1}/{EPOCHS}] "
                    f"[Val Step {step}/{len(reward_val_loader)}] "
                    f"Loss: {loss.item():.4f} | "
                    f"Chosen mean: {c.mean().item():.4f} | "
                    f"Rejected mean: {r.mean().item():.4f}"
                )

    val_loss = val_total / len(reward_val_loader)

    # -----------------------
    # EPOCH SUMMARY
    # -----------------------
    print("=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS} Summary")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val   Loss: {val_loss:.4f}")
    print("=" * 60)

    # -----------------------
    # SAVE BEST MODEL
    # -----------------------
    if val_loss < best_val:
        best_val = val_loss
        torch.save(reward_model.state_dict(), "best_reward_model.pt")
        print("✅ Saved new best reward model\n")

[Epoch 1/1] [Train Step 0/500] Loss: 1.7673 | Chosen mean: -6.0447 | Rejected mean: -4.5541
[Epoch 1/1] [Train Step 10/500] Loss: 0.9723 | Chosen mean: -4.3741 | Rejected mean: -4.2109
[Epoch 1/1] [Train Step 20/500] Loss: 0.7445 | Chosen mean: -5.6641 | Rejected mean: -5.5800
[Epoch 1/1] [Train Step 30/500] Loss: 0.6606 | Chosen mean: -5.1401 | Rejected mean: -5.4526
[Epoch 1/1] [Train Step 40/500] Loss: 1.0276 | Chosen mean: -5.0590 | Rejected mean: -4.4871
[Epoch 1/1] [Train Step 50/500] Loss: 0.8613 | Chosen mean: -4.8440 | Rejected mean: -4.5785
[Epoch 1/1] [Train Step 60/500] Loss: 0.7883 | Chosen mean: -4.7168 | Rejected mean: -4.6736
[Epoch 1/1] [Train Step 70/500] Loss: 0.7396 | Chosen mean: -4.8548 | Rejected mean: -4.8698
[Epoch 1/1] [Train Step 80/500] Loss: 0.5781 | Chosen mean: -5.4190 | Rejected mean: -5.8081
[Epoch 1/1] [Train Step 90/500] Loss: 0.9217 | Chosen mean: -6.0151 | Rejected mean: -5.6260
[Epoch 1/1] [Train Step 100/500] Loss: 0.5730 | Chosen mean: -5.3896 | 

In [18]:
ppo_policy = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")
ppo_ref = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")

for p in ppo_ref.parameters():
    p.requires_grad=False

In [19]:
def prepare_ppo(example):

    tokens = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

    return {
        "input_ids":tokens.input_ids.squeeze(0),
        "attention_mask":tokens.attention_mask.squeeze(0)
    }

In [20]:
ppo_train_dataset = train_pref.map(
    prepare_ppo,
    remove_columns=["prompt","chosen","rejected"]
)

ppo_val_dataset = val_pref.map(
    prepare_ppo,
    remove_columns=["prompt","chosen","rejected"]
)

In [21]:
class PPOCollator:

    def __call__(self,batch):

        return {

            "input_ids":torch.stack([torch.tensor(x["input_ids"]) for x in batch]),
            "attention_mask":torch.stack([torch.tensor(x["attention_mask"]) for x in batch])
        }

In [22]:
ppo_train_loader = DataLoader(
    ppo_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=PPOCollator()
)

ppo_val_loader = DataLoader(
    ppo_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=PPOCollator()
)

In [25]:
optimizer = torch.optim.AdamW(ppo_policy.parameters(), lr=1e-6)

beta = 0.02
clip_eps = 0.2

for epoch in range(EPOCHS):

    # -------------------
    # TRAIN
    # -------------------
    ppo_policy.train()

    total_loss = 0
    total_reward = 0
    total_kl = 0

    for step, batch in enumerate(ppo_train_loader):

        prompts = batch["input_ids"].to(device)
        prompt_mask = batch["attention_mask"].to(device)

        generated, gen_mask = generate_response(ppo_policy, tokenizer, prompts, prompt_mask)

        generated = generated.to(device)
        gen_mask = gen_mask.to(device)

        reward = reward_model(generated, gen_mask)

        logp = compute_logprob(ppo_policy, generated, gen_mask)
        logp_ref = compute_logprob(ppo_ref, generated, gen_mask)

        ratio = torch.exp(logp - logp_ref)
        clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)

        policy_loss = -torch.min(ratio * reward, clipped * reward)
        kl = (logp - logp_ref)

        loss = policy_loss + beta * kl

        optimizer.zero_grad()
        loss.mean().backward() # Fix: Call .mean() to get a scalar loss
        optimizer.step()

        total_loss += loss.mean().item()
        total_reward += reward.mean().item()
        total_kl += kl.mean().item()

        if step % 10 == 0:
            print(
                f"[Epoch {epoch+1}/{EPOCHS}] "
                f"[Train Step {step}/{len(ppo_train_loader)}] "
                f"Loss: {loss.mean().item():.4f} | "
                f"Reward: {reward.mean().item():.4f} | "
                f"KL: {kl.mean().item():.4f} | "
                f"Ratio: {ratio.mean().item():.4f}"
            )

    train_loss = total_loss / len(ppo_train_loader)
    train_reward = total_reward / len(ppo_train_loader)
    train_kl = total_kl / len(ppo_train_loader)

    # -------------------
    # VALIDATION
    # -------------------
    ppo_policy.eval()

    val_reward_total = 0
    val_kl_total = 0

    with torch.no_grad():

        for step, batch in enumerate(ppo_val_loader):

            prompts = batch["input_ids"].to(device)
            prompt_mask = batch["attention_mask"].to(device)

            generated, _ = generate_response(ppo_policy, tokenizer, prompts)

            generated = generated.to(device)
            generated, gen_mask = generate_response(ppo_policy, tokenizer, prompts, prompt_mask)
            # Create attention_mask for full generated sequence
            gen_mask = torch.ones_like(generated[:, MAX_LENGTH:])
            gen_mask = gen_mask.to(device)

            reward = reward_model(generated, gen_mask)

            logp = compute_logprob(ppo_policy, generated, gen_mask)
            logp_ref = compute_logprob(ppo_ref, generated, gen_mask)

            kl = (logp - logp_ref)

            val_reward_total += reward.mean().item()
            val_kl_total += kl.mean().item()

            if step % 10 == 0:
                print(
                    f"[Epoch {epoch+1}/{EPOCHS}] "
                    f"[Val Step {step}/{len(ppo_val_loader)}] "
                    f"Reward: {reward.mean().item():.4f} | "
                    f"KL: {kl.mean().item():.4f}"
                )

    val_reward = val_reward_total / len(ppo_val_loader)
    val_kl = val_kl_total / len(ppo_val_loader)

    # -------------------
    # EPOCH SUMMARY
    # -------------------
    print("=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS} Summary")
    print(f"Train Loss   : {train_loss:.4f}")
    print(f"Train Reward : {train_reward:.4f}")
    print(f"Train KL     : {train_kl:.4f}")
    print(f"Val Reward   : {val_reward:.4f}")
    print(f"Val KL       : {val_kl:.4f}")
    print("=" * 60)


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


[Epoch 1/1] [Train Step 0/500] Loss: 4.1409 | Reward: -5.5433 | KL: -14.6877 | Ratio: 0.0509


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

[Epoch 1/1] [Train Step 10/500] Loss: 4.4563 | Reward: -5.4907 | KL: -18.8620 | Ratio: 0.3189


...

[Epoch 1/1] [Train Step 480/500] Loss: 4.0550 | Reward: -5.8692 | KL: -32.0213 | Ratio: 0.0004


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

[Epoch 1/1] [Train Step 490/500] Loss: 31.6961 | Reward: -5.2237 | KL: -9.2680 | Ratio: 5.8502


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

TypeError: generate_response() missing 1 required positional argument: 'attention_mask'

In [ ]:
test_prompt = "What is the meaning of life?"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

# Generate from PPO Policy
ppo_policy.eval()
ppo_output = generate_response(ppo_policy, tokenizer, inputs.input_ids, inputs.attention_mask)[0]
ppo_text = tokenizer.decode(ppo_output[0], skip_special_tokens=True)

# Generate from SFT Model for comparison
sft_model.eval()
sft_output = generate_response(sft_model, tokenizer, inputs.input_ids, inputs.attention_mask)[0]
sft_text = tokenizer.decode(sft_output[0], skip_special_tokens=True)

print(f"Prompt: {test_prompt}\n")
print(f"SFT Model Response:\n{sft_text[len(test_prompt):].strip()}\n")
print(f"PPO Policy Response:\n{ppo_text[len(test_prompt):].strip()}")